In [1]:

# %%
# ==============================================
# Powercast — Week 3 Section 3 (Unified, Single Cell)
# Evaluation, Model Interpretation & Insights (Business)
# - PROFILE support: dev | preprod | final
# - Diagnostics: correlations, SARIMAX residual bias, XGB lag importances
# - Frequency-aware; runtime-capped; clean prints
# - Writes to <BASE_DIR>/results/Wk03_Section3_<profile>/{reports,plots}
# - Upserts Section 3 into consolidated Week 3 report (no duplicates)
# ==============================================

# ---------- Top-level configuration ----------
PROFILE = "dev"        # dev | preprod | final
DO = {                 # toggle diagnostics here
    "correlations": True,
    "sarimax":      True,
    "xgb":          True,
    "residuals":    True,   # residual bias by hour (for any model we forecast)
}

# ---------- Safety & stability ----------
from pathlib import Path
import os, re, sys, math, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

# headless plotting
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Cap threads to avoid BLAS/OMP storms
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Optional libs
_HAVE_SM = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    _HAVE_SM = True
except Exception:
    _HAVE_SM = False

_HAVE_XGB = False
try:
    from xgboost import XGBRegressor as _XGB
    def XGBRegressor(*args, **kwargs):
        kwargs.setdefault("n_estimators", 200 if PROFILE=="dev" else 400)
        kwargs.setdefault("max_depth", 4)
        kwargs.setdefault("learning_rate", 0.1)
        kwargs.setdefault("subsample", 0.9)
        kwargs.setdefault("colsample_bytree", 0.9)
        kwargs.setdefault("tree_method", "hist")
        kwargs.setdefault("n_jobs", 1)
        return _XGB(*args, **kwargs)
    _HAVE_XGB = True
except Exception:
    _HAVE_XGB = False

# ---------- Paths & profile envelope ----------
BASE_DIR = Path.cwd()
if BASE_DIR.name.lower() == "code":
    BASE_DIR = BASE_DIR.parent

SECTION_ID = "Wk03_Section3"
CFG = {
    "dev":     dict(max_days=90,   zones=1, test_days=7),
    "preprod": dict(max_days=365,  zones=3, test_days=28),
    "final":   dict(max_days=None, zones=3, test_days=28),
}
cfg = CFG.get(PROFILE, CFG["dev"])

RESULTS_DIR = BASE_DIR / "results" / f"{SECTION_ID}_{PROFILE}"
REPORTS_DIR = RESULTS_DIR / "reports"
PLOTS_DIR   = RESULTS_DIR / "plots"
for d in (REPORTS_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATA_PATH    = BASE_DIR / "data" / "Tetuan City power consumption.csv"
CONSOLIDATED = BASE_DIR / "SDS-CP036-powercast_Wk03_Report_Business.md"

def rel(p: Path) -> str:
    try: return p.relative_to(BASE_DIR).as_posix()
    except Exception: return p.as_posix()

print(f"[BOOT] BASE_DIR={BASE_DIR}")
print(f"[CFG ] PROFILE={PROFILE} zones={cfg['zones']} max_days={cfg['max_days']} test_days={cfg['test_days']}")
print(f"[LIBS] statsmodels={_HAVE_SM} xgboost={_HAVE_XGB}")
print(f"[OUT ] {RESULTS_DIR}")

# ---------- Helpers ----------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [re.sub(r"\s+", " ", c.strip()) for c in df.columns]
    return df

def infer_step_and_freq(idx: pd.DatetimeIndex):
    try:
        freq = pd.infer_freq(idx[:1000])
    except Exception:
        freq = None
    if freq:
        step = pd.to_timedelta(pd.tseries.frequencies.to_offset(freq))
        return step, freq
    diffs = pd.Series(idx).diff().dropna()
    step = diffs.mode().iloc[0] if not diffs.empty else pd.Timedelta(hours=1)
    total_sec = int(step.total_seconds())
    if total_sec % 3600 == 0:
        freq = f"{total_sec // 3600}H"
    elif total_sec % 60 == 0:
        freq = f"{total_sec // 60}T"
    else:
        freq = f"{total_sec}S"
    return step, freq

def limit_last_days(df: pd.DataFrame, date_col="DateTime", max_days=None) -> pd.DataFrame:
    if not max_days:
        return df
    df = df.copy()
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(date_col)
        cutoff = df[date_col].max() - pd.Timedelta(days=max_days)
        df = df[df[date_col] >= cutoff]
    return df

def pick_zone_cols(df: pd.DataFrame, k: int):
    cols = [c for c in df.columns if re.match(r"Zone\s+\d+\s+Power\s+Consumption", c)]
    cols = sorted(cols, key=lambda s: int(re.search(r"\d+", s).group()))
    return cols[:k] if k else cols

def make_time_splits(df: pd.DataFrame, test_days: int, date_col="DateTime", steps_per_day: int = 24):
    df = df.sort_values(date_col)
    test_len = int(test_days * steps_per_day)
    if test_len >= len(df):
        test_len = max(1, len(df)//10)
    test = df.iloc[-test_len:].copy()
    train_val = df.iloc[:-test_len].copy()
    if len(train_val) > 100:
        cut = int(len(train_val) * 0.85)
        train, val = train_val.iloc[:cut].copy(), train_val.iloc[cut:].copy()
    else:
        train, val = train_val.copy(), pd.DataFrame(columns=train_val.columns)
    return train, val, test

def metrics_series(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    denom = np.maximum(1e-8, np.where(y_true==0, 1e-8, y_true))
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    return dict(rmse=rmse, mae=mae, mape=mape)

def save_bar(series: pd.Series, title: str, path: Path):
    plt.figure(figsize=(8,4))
    series.plot(kind="bar")
    plt.title(title); plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close("all")

def save_plot_ts(ts_df: pd.DataFrame, cols, title, path: Path):
    plt.figure(figsize=(10,4))
    for c in cols:
        plt.plot(ts_df.index, ts_df[c], label=c)
    plt.title(title)
    plt.xlabel("Time"); plt.ylabel("Power")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close("all")

# ---------- Load & prepare ----------
if not DATA_PATH.exists():
    msg = f"Data file not found: {rel(DATA_PATH)}"
    print("[ERROR]", msg)
    (REPORTS_DIR / "SDS-CP036-powercast_Wk03_Section3_Business_Report.md").write_text(
        f"# SDS-CP036-powercast – Week 3 Section 3: Evaluation & Model Interpretation & Insights\n\n"
        f"Profile: `{PROFILE}`\n\n"
        f"**Error:** {msg}\n", encoding="utf-8"
    )
    raise SystemExit(0)

raw = pd.read_csv(DATA_PATH)
raw = normalize_columns(raw)

if "DateTime" not in raw.columns:
    alt = [c for c in raw.columns if c.lower()=="datetime"]
    if alt:
        raw = raw.rename(columns={alt[0]:"DateTime"})
    else:
        raise RuntimeError("Missing DateTime column. Got: " + ", ".join(raw.columns))

raw = limit_last_days(raw, "DateTime", cfg["max_days"])

# Exogenous columns (case/space tolerant matching)
def find_col(candidates):
    out = []
    lower = {c.lower(): c for c in raw.columns}
    for name in candidates:
        key = name.lower()
        # exact lower
        if key in lower: out.append(lower[key]); continue
        # regex like
        for c in raw.columns:
            if re.fullmatch(key.replace(" ","\s+"), c, flags=re.IGNORECASE):
                out.append(c); break
    return out

exo_candidates = ["Temperature","Humidity","Wind Speed","general diffuse flows","diffuse flows"]
zones = pick_zone_cols(raw, cfg["zones"])
if not zones:
    raise RuntimeError("No 'Zone X Power Consumption' columns found. Got: " + ", ".join(raw.columns))
exos = find_col(exo_candidates)

work = raw[["DateTime"] + zones + exos].copy()
work["DateTime"] = pd.to_datetime(work["DateTime"])
work = work.sort_values("DateTime").set_index("DateTime")

step_td, freq_str = infer_step_and_freq(work.index)
steps_per_hour = max(1, int(round(pd.Timedelta(hours=1) / step_td)))
steps_per_day  = int(24 * steps_per_hour)
print(f"[FREQ] step={step_td} freq='{freq_str}' steps/hour={steps_per_hour} steps/day={steps_per_day}")

train, val, test = make_time_splits(work.reset_index(), cfg["test_days"], "DateTime", steps_per_day=steps_per_day)
train = train.set_index("DateTime"); val = val.set_index("DateTime"); test = test.set_index("DateTime")

# ---------- 1) Correlations (train window) ----------
corr_rows = []
if DO["correlations"]:
    for zone in zones:
        s = pd.to_numeric(train[zone], errors="coerce")
        for ex in exos:
            r = float(pd.concat([s, pd.to_numeric(train[ex], errors="coerce")], axis=1).corr().iloc[0,1])
            corr_rows.append({"zone": zone, "feature": ex, "pearson": r})
        # plot per zone
        if exos:
            ser = pd.Series({ex: next((r["pearson"] for r in corr_rows if r["zone"]==zone and r["feature"]==ex), np.nan) for ex in exos})
            pth = PLOTS_DIR / f"{zone.replace(' ','_')}_feature_correlations.png"
            save_bar(ser.sort_values(ascending=False), f"{zone} — Feature correlations (train)", pth)
feature_corr_csv = REPORTS_DIR / "feature_correlations.csv"
pd.DataFrame(corr_rows).to_csv(feature_corr_csv, index=False)

# ---------- 2) SARIMAX (hourly, lightweight) ----------
sarimax_coef_rows = []
sarimax_forecasts = {}
if DO["sarimax"] and _HAVE_SM:
    for zone in zones:
        # hourly resample, small cap
        series_hr = pd.to_numeric(pd.concat([train[zone], val[zone]]), errors="coerce").ffill().resample("1H").mean().asfreq("H").ffill()
        if len(series_hr) > 2400:
            series_hr = series_hr.iloc[-2400:]
        model = SARIMAX(series_hr, order=(1,0,1), seasonal_order=(0,1,1,24),
                        enforce_stationarity=False, enforce_invertibility=False,
                        simple_differencing=True, trend=None, concentrate_scale=True)
        res = model.fit(method="lbfgs", maxiter=40, disp=False)
        # coefficients
        for name, coef_val in res.params.items():
            sarimax_coef_rows.append({"zone": zone, "param": name, "value": float(coef_val)})
        # forecast native by down/up-sampling
        h = int(math.ceil(len(test) / steps_per_hour))
        fc_hr = res.get_forecast(steps=h)
        yhat_hr = pd.Series(fc_hr.predicted_mean.values.astype(float),
                            index=pd.date_range(series_hr.index[-1] + pd.Timedelta(hours=1), periods=h, freq="H"))
        native_hours = pd.to_datetime(test.index.floor("H"))
        yhat_native = yhat_hr.reindex(native_hours, method="ffill").values[:len(test)]
        sarimax_forecasts[zone] = yhat_native
        # plot actual vs pred
        comp = pd.DataFrame({"actual": test[zone], "pred": yhat_native}, index=test.index)
        pth = PLOTS_DIR / f"{zone.replace(' ','_')}_sarimax_actual_vs_pred.png"
        save_plot_ts(comp, ["actual","pred"], f"{zone} — SARIMAX (hourly) Forecast", pth)

sarimax_coef_csv = REPORTS_DIR / "sarimax_coefficients.csv"
pd.DataFrame(sarimax_coef_rows).to_csv(sarimax_coef_csv, index=False)

# ---------- 3) XGBoost lag importances ----------
xgb_imp_rows = []
xgb_forecasts = {}
if DO["xgb"] and _HAVE_XGB:
    for zone in zones:
        series = pd.to_numeric(pd.concat([train[zone], val[zone]]), errors="coerce").ffill()
        df_sup = pd.DataFrame({"y": series})
        max_lag = 24
        for L in range(1, max_lag+1):
            df_sup[f"lag_{L}"] = series.shift(L)
        df_sup = df_sup.dropna()
        # cap rows
        max_rows = 5000 if PROFILE=="dev" else 20000
        if len(df_sup) > max_rows:
            df_sup = df_sup.iloc[-max_rows:]
        X = df_sup.drop(columns=["y"]).values
        y = df_sup["y"].values
        model = XGBRegressor()
        model.fit(X, y, verbose=False)
        # importances
        imp = pd.Series(model.feature_importances_, index=[c for c in df_sup.columns if c!="y"])
        top10 = imp.sort_values(ascending=False).head(10)
        for k,v in imp.items():
            xgb_imp_rows.append({"zone":zone, "feature":k, "importance": float(v)})
        # plot
        pth = PLOTS_DIR / f"{zone.replace(' ','_')}_xgb_top_importances.png"
        save_bar(top10, f"{zone} — XGBoost lag importances (top 10)", pth)
        # recursive forecast
        hist = list(series.values)
        preds = []
        for _ in range(len(test)):
            feats = [hist[-k] for k in range(1, max_lag+1)]
            pred = float(model.predict(np.array(feats).reshape(1,-1)))
            preds.append(pred); hist.append(pred)
        xgb_forecasts[zone] = np.array(preds, dtype=float)

xgb_imp_csv = REPORTS_DIR / "xgb_importances.csv"
pd.DataFrame(xgb_imp_rows).to_csv(xgb_imp_csv, index=False)

# ---------- 4) Residual bias by hour ----------
bias_rows = []
if DO["residuals"]:
    for zone in zones:
        # prefer xgb if available else sarimax
        if zone in xgb_forecasts:
            yhat = xgb_forecasts[zone]
            model_tag = "XGBoost (lags)"
        elif zone in sarimax_forecasts:
            yhat = sarimax_forecasts[zone]
            model_tag = "SARIMAX (hourly)"
        else:
            continue
        actual = pd.to_numeric(test[zone], errors="coerce").values
        resid = actual - yhat
        hours = pd.to_datetime(test.index).hour
        dfb = pd.DataFrame({"hour": hours, "residual": resid})
        grp = dfb.groupby("hour")["residual"].mean().reindex(range(24)).fillna(0.0)
        for h, v in grp.items():
            bias_rows.append({"zone": zone, "model": model_tag, "hour": int(h), "mean_residual": float(v)})
        # plot residual bias
        pth = PLOTS_DIR / f"{zone.replace(' ','_')}_residual_bias_by_hour.png"
        save_bar(grp, f"{zone} — Residual bias by hour ({model_tag})", pth)

bias_csv = REPORTS_DIR / "residual_bias_by_hour.csv"
pd.DataFrame(bias_rows).to_csv(bias_csv, index=False)

# ---------- Build Section 3 business report ----------
section3_md = REPORTS_DIR / "SDS-CP036-powercast_Wk03_Section3_Business_Report.md"
report = f"""# SDS-CP036-powercast – Week 3 Section 3: Evaluation & Model Interpretation & Insights

Profile: `{PROFILE}`

### Key Questions Answered

**Q: How did you interpret feature importance or model coefficients, and what did they reveal about power consumption drivers?**  
**A:** We used three complementary lenses:
- **Correlation lens:** Pearson correlations between zone demand and drivers (Temperature, Humidity, Wind Speed, solar diffuse flows). See `{rel(feature_corr_csv)}` and bar charts in `{rel(PLOTS_DIR)}/`.
- **SARIMAX coefficients:** Lightweight daily-seasonal model on hourly data to expose autoregressive and seasonal strength. See `{rel(sarimax_coef_csv)}`.
- **XGBoost lag importances:** Non‑linear model with 24 lag features; top bars highlight which recent hours matter most. See `{rel(xgb_imp_csv)}` and plots in `{rel(PLOTS_DIR)}/`.

**Q: Did you observe any systematic errors or biases in your model predictions? How did you investigate and address them?**  
**A:** After forecasting the test window, we computed **residuals = Actual − Predicted** and averaged by **hour of day**. Consistent positive residuals at an hour mean we **under‑predict** that slot (peaks), negatives mean **over‑predict**. See `{rel(bias_csv)}` and the bias charts in `{rel(PLOTS_DIR)}/`.  
**Actions:** If evening hours are under‑predicted, we (a) enable native‑freq SARIMAX in `final` to capture sharper cycles, (b) add more lags or calendar/weather features to XGBoost.

**Q: What trade-offs did you consider when selecting your final model(s) for each zone?**  
**A:** We balance **accuracy vs. simplicity vs. speed**:
- **SARIMAX:** fast, explainable, strong daily cycle capture; may miss non‑linear spikes.
- **XGBoost (lags):** better for non‑linear patterns; requires feature care and is heavier than SARIMAX.
The chosen model per zone reflects the smallest footprint that meets accuracy needs, guided by the above diagnostics.

### What this run produced
- Correlation table: `{rel(feature_corr_csv)}`  
- SARIMAX coefficients: `{rel(sarimax_coef_csv)}`  
- XGBoost importances: `{rel(xgb_imp_csv)}`  
- Residual bias by hour: `{rel(bias_csv)}`  
- Plots: `{rel(PLOTS_DIR)}/`

### Business Value (Executive View)
- **Clarity on drivers:** Leaders see which **weather/time signals** move demand.  
- **Actionable fixes:** Bias‑by‑hour points to where to **tune models or operations** (staffing, procurement).  
- **Confidence & defensibility:** Simple visuals + tables support decisions in reviews and audits.
"""
section3_md.write_text(report, encoding="utf-8")
print(f"✅ Wrote section report: {rel(section3_md)}")

# ---------- Consolidated upsert (replace-or-append Section 3) ----------
def upsert_section_3(consolidated_path: Path, new_block: str):
    # ensure a base file exists
    if not consolidated_path.exists():
        base = [
            "# SDS-CP036-powercast – Week 3 (Business Report)",
            "",
            "## Table of Contents",
            "- [Section 1 – Model Selection & Training](#sds-cp036-powercast-–-week-3-section-1-model-selection--training)",
            "- [Section 2 – MLflow Experiment Tracking & Evaluation](#sds-cp036-powercast-–-week-3-section-2-mlflow-experiment-tracking--evaluation)",
            "- [Section 3 – Evaluation & Model Interpretation & Insights](#sds-cp036-powercast-–-week-3-section-3-evaluation--interpretation--insights)",
            ""
        ]
        consolidated_path.write_text("\n".join(base), encoding="utf-8")
    text = consolidated_path.read_text(encoding="utf-8")

    hdr_s1 = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*3\s*Section\s*1\s*:\s*Model\s*Selection\s*\&\s*Training\s*$")
    hdr_s2 = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*3\s*Section\s*2\s*:\s*MLflow\s*Experiment\s*Tracking\s*\&\s*Evaluation\s*$")
    hdr_s3 = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*3\s*Section\s*3\s*:\s*Evaluation\s*\&\s*Model\s*Interpretation\s*\&\s*Insights\s*$")
    hdr_any = re.compile(r"(?m)^\#\s*SDS-CP036-powercast\s*[–-]\s*Week\s*3\s*Section\s*[1-3]\s*:\s*.*$")

    # remove existing section 3 block
    def remove_block(txt, hdr_regex):
        m = hdr_regex.search(txt)
        if not m:
            return txt
        start = m.start()
        m2 = hdr_any.search(txt, pos=m.end())
        end = m2.start() if m2 else len(txt)
        return (txt[:start] + txt[end:]).strip()

    text = remove_block(text, hdr_s3).strip()

    # insert after S2 if exists, else after S1, else append
    m2 = hdr_s2.search(text)
    if m2:
        next_m = hdr_any.search(text, pos=m2.end())
        end2 = next_m.start() if next_m else len(text)
        new_text = text[:end2].rstrip() + "\n\n" + new_block.strip() + "\n\n" + text[end2:].lstrip()
    else:
        m1 = hdr_s1.search(text)
        if m1:
            next_m = hdr_any.search(text, pos=m1.end())
            end1 = next_m.start() if next_m else len(text)
            new_text = text[:end1].rstrip() + "\n\n" + new_block.strip() + "\n\n" + text[end1:].lstrip()
        else:
            new_text = text.rstrip() + "\n\n" + new_block.strip() + "\n"

    consolidated_path.write_text(new_text, encoding="utf-8")
    print(f"✅ Upserted Section 3 into consolidated: {rel(consolidated_path)}")

block = section3_md.read_text(encoding="utf-8").strip()
upsert_section_3(CONSOLIDATED, block)

# ---------- Tail preview ----------
try:
    print("\n--- Files created ---")
    for p in sorted(REPORTS_DIR.glob("*.csv")):
        print("CSV:", rel(p))
    for p in sorted(PLOTS_DIR.glob("*.png"))[:6]:
        print("PNG:", rel(p))
except Exception as e:
    print("Preview error:", e)

print("\nDone.")


[BOOT] BASE_DIR=/home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast
[CFG ] PROFILE=dev zones=1 max_days=90 test_days=7
[LIBS] statsmodels=True xgboost=True
[OUT ] /home/6376f5a9-d12b-4255-9426-c0091ad440a7/Powercast/results/Wk03_Section3_dev
[FREQ] step=0 days 00:10:00 freq='10T' steps/hour=6 steps/day=144
✅ Wrote section report: results/Wk03_Section3_dev/reports/SDS-CP036-powercast_Wk03_Section3_Business_Report.md
✅ Upserted Section 3 into consolidated: SDS-CP036-powercast_Wk03_Report_Business.md

--- Files created ---
CSV: results/Wk03_Section3_dev/reports/feature_correlations.csv
CSV: results/Wk03_Section3_dev/reports/residual_bias_by_hour.csv
CSV: results/Wk03_Section3_dev/reports/sarimax_coefficients.csv
CSV: results/Wk03_Section3_dev/reports/xgb_importances.csv
PNG: results/Wk03_Section3_dev/plots/Zone_1_Power_Consumption_feature_correlations.png
PNG: results/Wk03_Section3_dev/plots/Zone_1_Power_Consumption_residual_bias_by_hour.png
PNG: results/Wk03_Section3_dev/plots/Zone_1_Po